In [1]:
%load_ext autoreload
%autoreload 2

import sys
from importlib import import_module
sys.path.append("..")   # geocoding_ml/ -> lets us import from helpers/

import pandas as pd

from helpers.pairs import load_pairs, add_number_street
from helpers.features import get_processed_dfs
from helpers.check_model_performance import review_typical_slices, find_random_sample_given_mask
run_against_models = import_module("model_baseline.01_process_dumb_models").run_against_models

In [2]:
ot_train, ot_train_label = load_pairs("overture", "train")
ot_val, ot_val_label = load_pairs("overture", "val")
ot_test, ot_test_label = load_pairs("overture", "test")

# House number + street remainder for the similarity features (helpers/pairs.py):
# adds number_a/b, street_a/b, number_missing, both_number_missing
ot_train, ot_val, ot_test = [add_number_street(df) for df in [ot_train, ot_val, ot_test]]

In [3]:
# Full feature build (cached; ~1.5 min on first run)
train_overture, val_overture, test_overture = get_processed_dfs([
    [ot_train, "./overture_training/overture_evaluation_train.parquet"],
    [ot_val, "./overture_training/overture_evaluation_val.parquet"],
    [ot_test, "./overture_training/overture_evaluation_test.parquet"],
])

In [4]:
# Overture -> Overture, scored on val
ret = run_against_models(train_overture, ot_train_label, val_overture, test_overture)

In [5]:
ret.keys()

dict_keys(['LogisticRegression', 'RandomForestClassifier', 'XGBClassifier'])

In [6]:
for model in ['LogisticRegression', 'RandomForestClassifier', 'XGBClassifier']:
    print(f"{'='*20} {model} {'='*20}")
    print(review_typical_slices(ot_val, ret[model]["val"][0], ot_val_label))
    print()


==================== LogisticRegression ====================
                                      slice      n  correct  errors % correct
0                      positives: identical   3390     3389       1    100.0%
1                  positives: non-identical    797      652     145     81.8%
2     neg: case1 (same street, diff number)  13781    13753      28     99.8%
3  neg: case2 (same number, similar street)     34        1      33      2.9%
4                         neg: case3 (easy)  23863    23857       6    100.0%
5                        overall (accuracy)  41865    41652     213     99.5%

==================== RandomForestClassifier ====================
                                      slice      n  correct  errors % correct
0                      positives: identical   3390     3390       0    100.0%
1                  positives: non-identical    797      681     116     85.4%
2     neg: case1 (same street, diff number)  13781    13715      66     99.5%
3  neg: case2 (

In [7]:
sample = find_random_sample_given_mask(ot_val, ret["LogisticRegression"]["val"][0], ot_val_label, 0)
print(sample)

==========positives: identical==========
                                       id_a  \
7198   5868b3c4-0602-4620-9e88-0f18797b9450   
3201   84197d30-5ef7-4802-a2ca-e46cbd2a59f5   
3184   4c9f6182-fc0d-486e-878f-80ce0cbc17f9   
20248  bdbf5611-3a3b-4457-a77f-d92c5352bf65   
9071   87e10df9-26e3-4981-8257-57d0b92e1cef   
8432   1bd5778c-2503-4066-b8ff-375ba4eda710   
19799  9333ade1-7436-488a-810e-db978dc8582e   
3371   030a63df-24ec-4e3a-bcd6-a79169f7fd5d   
11799  e584e858-1810-43a5-b89f-6f06a5847601   
2773   07aa789b-e139-4e5a-8f27-03aadcb39b6e   

                                       id_b           address_a  \
7198   5868b3c4-0602-4620-9e88-0f18797b9450    221 BULL HILL LN   
3201   84197d30-5ef7-4802-a2ca-e46cbd2a59f5   565 LONG HILL AVE   
3184   4c9f6182-fc0d-486e-878f-80ce0cbc17f9  484 BRIDGEPORT AVE   
20248  bdbf5611-3a3b-4457-a77f-d92c5352bf65         80 DOYLE RD   
9071   87e10df9-26e3-4981-8257-57d0b92e1cef      1799 SUMMER ST   
8432   1bd5778c-2503-4066-b8ff-375ba4ed

### Results — Overture → Overture, val (refreshed 2026-10-16)

23 features (22 similarities + `number_missing`), default hyperparameters, no class weighting. Numbers are **% of the slice handled correctly** on val (found for matches, rejected for non-matches), as printed by the `review_typical_slices` cell above. Test numbers and slice sizes: `results.md` (repo root).

| model | identical matches found | **non-identical matches found** | case1 rejected | case2 rejected | easy rejected |
|---|---|---|---|---|---|
| LogReg | 100.0 | **81.8** | 99.8 | 2.9 | 100.0 |
| RF | 100.0 | **85.4** | 99.5 | 23.5 | 100.0 |
| XGBoost | 100.0 | **84.9** | 99.5 | 20.6 | 100.0 |

- The old headline F1 0.97 was inflated by identical-string positives (~79% of positives, ~all found); the non-identical column is the one that matters.
- **Case 2** is only 34 val rows and contains probable label noise (open question, hand-check pending) — don't interpret it yet.
- **Visible error sources:** text before the number on side B → `number_missing`; `ST`/`STREET` suffix variants.

*Diagnostic queries + results for this section: **Diagnostics** at the bottom of this notebook → Overture baseline.*

## Diagnostics

One-off checks kept out of the pipeline above. Paths are relative to this notebook's folder.

### House-number regex

**Problem:** The old `^(\d+)\s+` pattern missed numbers like `12A` and `16-18`, leaving them in the street name.  
**Check:** New regex on hand-picked cases; streets still starting with a digit; raw addresses the old pattern missed.  
**Result:** Hand cases correct; 473 / 492 streets start with a digit (ordinals, fractions, double numbers); old pattern missed 1,202 / 1,484 letter suffixes and 444 / 742 ranges (A / B) (re-run 2026-10-13)

```python
# Test the house-number token regex on hand-picked cases
check = pd.DataFrame({"address": ["12 MAIN ST", "12A MAIN ST", "16-18 ELM ST", "20-A ELM ST"]})
check["check"] = check["address"].str.extract(r"^(\d[A-Z0-9a-z-]*)\s")
print(check)

# After parsing: sample + count of streets that still start with a digit (ordinal streets, 1/2 addresses, double numbers)
ret = pd.concat([ot_train, ot_val, ot_test])
print(ret[ret["street_a"].str.match(r"^\d")][["street_a"]].sample(frac=1).head(10))
print(ret["street_a"].str.match(r"^\d").sum())
print(ret[ret["street_b"].str.match(r"^\d")][["street_b"]].sample(frac=1).head(10))
print(ret["street_b"].str.match(r"^\d").sum())

# Raw addresses the old ^(\d+)\s+ pattern missed: letter suffix (12A) on A / B, then ranges (16-18) on A / B
print(ret["address_a"].str.match(r"^\d+[A-Za-z]+\s").sum())
print(ret["address_b"].str.match(r"^\d+[A-Za-z]+\s").sum())
print(ret["address_a"].str.match(r"^[0-9]+-[0-9]+\s").sum())
print(ret["address_b"].str.match(r"^[0-9]+-[0-9]+\s").sum())
```

### Streets under 2 characters

**Problem:** The Overlap bigram metric divides by zero on strings shorter than 2 characters.  
**Check:** Find any short street names.  
**Result:** Test split only: street `1` (7 on A, 11 on B) → guarded in `features.py` (re-run 2026-10-13)

```python
# Streets under 2 characters (break bigram metrics): only the test addresses that are literally "1"
for df in [ot_train, ot_val, ot_test]:
    print(df[df["street_a"].str.len() < 2]["street_a"])
    print(df[df["street_b"].str.len() < 2]["street_b"])
    print()
```

### Positives with identical addresses

**Problem:** Suspected the models were misclassifying positives whose addresses are identical.  
**Check:** Misclassifications within `case0`, split by identical vs. different address and by number status.  
**Result:** Identical: LR 1 / 3,390 wrong (a ZIP disagreement), RF 0, XGB 0. Different: 82–85% correct. The `number_missing` subset is empty by construction since `both_number_missing` was split out; use `both_number_missing` instead (re-run 2026-10-13)

```python
print([k for k in ret.keys()])
mask = ((ot_val["neg_type"] == "case0") & (ot_val["address_a"] == ot_val["address_b"])).values
totals = [(preds[mask] != ot_val_label[mask]).sum() for model, (preds, probs) in ret.items()]
print([int(c) for c in totals])

# missing = [ot_val[mask][(preds != ot_val_label) & (ot_val["number_missing"] == False)] for model, (preds, probs) in ret.items()]
# for miss,model in zip(missing,[k for k in ret.keys()]):
#     print(f"{'='*20} {model} {'='*20}")
#     print(miss[["address_a", "address_b", "ZipCode_a", "ZipCode_b"]])


# mask = ((ot_val["neg_type"] == "case0") & (ot_val["address_a"] != ot_val["address_b"])).values
# totals = [(preds[mask] == ot_val_label[mask]).sum() for model, (preds, probs) in ret.items()]
# print([int(c) for c in totals])

# mask = ((ot_val["neg_type"] == "case0") & 
#         (ot_val["address_a"] == ot_val["address_b"]) & 
#         (ot_val['number_missing'] == True)).values
# n = mask.sum()
# totals = [(preds[mask] != ot_val_label[mask]).sum() for model, (preds, probs) in ret.items()]
# print([float(c/n) for c in totals])

# mask = ((ot_val["neg_type"] == "case0") & 
#         (ot_val["address_a"] == ot_val["address_b"]) & 
#         (ot_val['number_missing'] == False)).values
# n = mask.sum()
# totals = [(preds[mask] != ot_val_label[mask]).sum() for model, (preds, probs) in ret.items()]
# print([float(c/n) for c in totals])

# print(
# len(ot_val[
#     (ot_val["neg_type"] == "case0") & 
#     (ot_val["address_a"] == ot_val["address_b"]) & 
#     (ot_val['number_missing'] == True)
# ]))
```